In [80]:
# !pip install -U duckduckgo-search

### Step 2: Setup

In [81]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from pprint import pprint
import json
from ddgs import DDGS
import trafilatura
from IPython.display import Markdown, display


load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("OPENAI_API_KEY environment variable is not set.")
else:
    print("OPENAI_API_KEY is set.")


client = OpenAI(api_key=OPENAI_API_KEY)

MODEL = "gpt-4.1-mini"

OPENAI_API_KEY is set.


### Step 3: Define the tools

In [82]:
def search_web(query: str):
    """Search the web using duckduckgo browser. Return 3 results"""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f"\u2705 Got results")
    return json.dumps(results, indent=2)

In [83]:
def fetch_url(url: str):
    """Fetching the content from url using trafilatura"""
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f"\u2705 Got text: {len(text)} characters")
            return text
    print(f"\u274c Failed to extract text from {url}.")
    return f"Could not extract text from {url}. Try a diffrent source."

In [84]:
# Test search 
search_web("Apples in Madagascar")

✅ Got results


'[\n  {\n    "title": "Apple Blossoms in Madagascar: A Botanical Anomaly and ...",\n    "href": "https://frameandfocal.com/photography-glossary/apple-blossoms-in-madagascar",\n    "body": "Aug 21, 2026 \\u00b7 Apple blossoms don\\u2019t naturally grow in Madagascar\\u2014but small-scale orchards exist. This article details their origins, climate constraints, bloom timing (Sept\\u2013Oct), and precise photographic techniques for capturing them under humid tropical conditions."\n  },\n  {\n    "title": "Field Guide to Fruit on Madagascar - jurgenenkatja",\n    "href": "https://www.jurgenenkatja.nl/en/fruit-on-madagascar/",\n    "body": "Oct 24, 2023 \\u00b7 This species originates from the Comoros and Madagascar. We came across this fruit on the east coast, but it also seems to grow, to a lesser extent, in the northwest."\n  },\n  {\n    "title": "The list of fruits to eat in Madagascar - explurt.com",\n    "href": "https://explurt.com/blog-english/madagascar-english/the-list-of-fruits-t

In [85]:
result = fetch_url("https://en.wikipedia.org/wiki/Annona_squamosa")
print(result)

✅ Got text: 15348 characters
Annona squamosa
| Annona squamosa |  | 
|---|---|
| Cross section of the fruit shown on right |  | 
| Scientific classification |  | 
| Kingdom: | Plantae | 
| Clade: | Tracheophytes | 
| Clade: | Angiosperms | 
| Clade: | Magnoliids | 
| Order: | Magnoliales | 
| Family: | Annonaceae | 
| Genus: | Annona | 
| Species: | A. squamosa | 
| Binomial name |  | 
| Annona squamosa |  | 
| Synonyms[3] |  | 
|  |  | 
Annona squamosa is a small, well-branched tree or shrub from the family Annonaceae that bears edible fruits called sugar apples, sweetsops or custard apples.[4][5][6] It tolerates a tropical lowland climate better than its relatives Annona reticulata and Annona cherimola,[7] and is the most widely cultivated of these Annona species.[5][8]
Annona squamosa is semi-(or late) deciduous,[9] and 3 to 8 metres (10 to 26 feet) tall,[4][9] similar to soursop (Annona muricata).[10] It is a native of tropical climate in the Americas, and Spanish traders aboard th

### Step 4: Describe as LLM tools

In [86]:
tools = []

In [87]:
search_web_function = {
    "name": "search_web",
    "description": "Search the web using duckduckgo browser. Returns 3 results",
    "parameters": {
        "type": "object",
        "properties":{
            "query":{
                "type": "string",
                "description": "The search query to find relevant websites"
            }
        },
        "required": ["query"]
    }
}

# Add pushover to the list of tools for LLM
tools.append({"type": "function", "function": search_web_function})

In [88]:
fetch_url_function = {
    "name": "fetch_url",
    "description": "Fetch and extract the amin text content from a web page.",
    "parameters": {
        "type": "object",
        "properties":{
            "url":{
                "type": "string",
                "description": "The URL of the web page to fetch and extract text from."
            }
        },
        "required": ["url"]
    }
}

# Add pushover to the list of tools for LLM
tools.append({"type": "function", "function": fetch_url_function})

In [89]:
# tools

### Step 5: Tool call handler

In [90]:
def handle_tool_call(tool_calls):
    tool_result = []
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        print(f" \U0001f527 Calling function {function_name}, with arguments{args}")
        
        # Route to the appropriate function based on the function name
        if function_name == "search_web":
            # Actually send the notification
            result = search_web(args["query"])
            content = f"Search result: {result}"
        elif function_name == "fetch_url":
            result = fetch_url(args["url"])
            content = f"Fetched URL content: {result}"
        
        else:
            content = f"Unknown funtion: {function_name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        }

        tool_result.append(tool_call_result)

    return tool_result

In [94]:
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word "DONE:" is a control signal, not a label. Never use it as a heading, section marker, or inline annotation. 
ONLY use the word "DONE:" as per the instructions below -- it has to come at the start of a reply.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 6 different sources, synthesize into a research brief

You MUST gather information from at least 3 distinct sources before delivering your brief. 
If you have fewer than 3 sources, keep searching.

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.
It is imparetive that "DONE:" should be at the start of the final response, so that it can be easily parsed and extracted.
You CANNOT and SHOULD NOT include "DONE:" in any other part of your response except at the very BEGINING of the final research brief

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

### Step 6: Agentic loop 

In [95]:
def run_research_agent(topic: str, max_iterations: int=10) -> str:
    """
    Run research agent on a topic and return the research brief
    
    Args:
    topic - the topic to research
    max_iterations - safety limit to prevent infinite loops

    returns:
    The research brief as a string.
    """
    print(f"\n\U0001F50D Starting research on {topic}")
    print("=" *60)

    messages = [
        {"role": "system", "content": RESEARCH_AGENT_PROMPT},
        {"role": "user", "content": f"Research the following topic and produce a comprehensive research brief: \n{topic}"}
    ]

    iteration = 0
    while iteration < max_iterations:
        iteration += 1 # just adds one to the iteration and the iteration starts with 1. 
        print(f"\n ---- Iteration {iteration} -----")

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
        )

        message = response.choices[0].message
        messages.append(message)

        if message.tool_calls:
            tool_results = handle_tool_call(message.tool_calls)
            messages.extend(tool_results)

        else:
            content = message.content # No tool calls just a normal text from LLM
            if content.startswith("DONE:"):
                research_brief = content[len("DONE:"):].strip()
                print(f"\n u2705 Research complete!")
                return research_brief
            else:
                print(f"  \U0001F4AD Agent is thinking: ")
                pprint(content)

        if (iteration == max_iterations - 1):
            print(f"   \u26a0 Safety limit reached. Stopping research in next iteration")
            messages.append({"role": "user", "content": "You have reached the maximum number of iterations. Please deliver your research brief now. You MUST respond with DONE: followed by your brief"})

    return "Research incomplete. Maximum iterations reached without finalizing research brief"


### Lets have it!


In [96]:
brief = run_research_agent("Best PHD programs in Remote sensing in the United States")
display(Markdown(brief))


🔍 Starting research on Best PHD programs in Remote sensing in the United States

 ---- Iteration 1 -----


h2 connection driver error: connection aborted
h2 connection driver error: connection aborted
h2 connection driver error: connection aborted
h2 connection driver error: connection aborted
h2 connection driver error: connection aborted
h2 connection driver error: connection aborted


 🔧 Calling function search_web, with arguments{'query': 'best PhD programs in remote sensing in the United States'}
✅ Got results

 ---- Iteration 2 -----
 🔧 Calling function fetch_url, with arguments{'url': 'https://edurank.org/engineering/remote-sensing/na/'}
✅ Got text: 10840 characters
 🔧 Calling function fetch_url, with arguments{'url': 'https://www.geologyprograms.com/rankings/best-remote-sensing-programs/'}
✅ Got text: 11868 characters
 🔧 Calling function fetch_url, with arguments{'url': 'https://elitedegrees.net/remote-sensing/united-states/'}
✅ Got text: 855 characters

 ---- Iteration 3 -----
  💭 Agent is thinking: 
('I have gathered detailed ranking data and insights on top PhD programs in '
 'remote sensing in the U.S. from three respected sources. I will now '
 'synthesize the information into a comprehensive research brief highlighting '
 'key facts, notable universities, program features, and additional pertinent '
 'data.\n'
 '\n'
 'DONE: \n'
 '\n'
 'Research Brief: Bes

Research Brief: Best PhD Programs in Remote Sensing in the United States

Overview:
Remote sensing is an advanced scientific field focused on acquiring, processing, and analyzing data from satellites and aerial platforms to understand earth systems. PhD programs in this area emphasize in-depth technical training in sensor physics, geospatial analysis, programming, and accuracy assessment.

Key Facts and Statistics:
- Remote sensing-related occupations, such as cartographers and photogrammetrists, are projected to grow 6% by 2034, double the average across occupations.
- Median wage for geoscientists (which includes remote sensing) was around $99,240 as of May 2024.
- PhD programs require strong programming skills, particularly in Python with libraries like rasterio and GDAL.
- The field demands access to imagery archives, processing infrastructure, and interdisciplinary collaboration.

Top Remote Sensing PhD Programs in the United States (based on research citations, faculty strength, program quality):

1. California Institute of Technology (Caltech)
   - Acceptance rate: 3%
   - Leading research in sensor physics and earth observation.

2. University of Maryland - College Park
   - Department of Geographical Sciences offers remote sensing PhD concentration.
   - Acceptance: 45%

3. University of Arizona
   - PhD in Geography with remote sensing focus.
   - Known for earth observation research.

4. University of Colorado Boulder
   - Strong Geography department with remote sensing emphasis.

5. Massachusetts Institute of Technology (MIT)
   - Focus on remote sensing for environmental and engineering applications.
   - Acceptance: 5%

6. University of Washington - Seattle
7. Pennsylvania State University
8. Texas A&M University - College Station
9. University of California campuses (Berkeley, Santa Barbara, Davis)
10. Stanford University
11. University of Wisconsin - Madison
12. University of Michigan - Ann Arbor

Program Themes:
- In-depth study of sensor technologies, image processing, and accuracy assessment including confusion matrices.
- Requirements for proficiency in programming for large scale data analysis.
- Research environment rich with access to sensor data and HPC infrastructure.
- Interdisciplinary projects combined with Environmental Science, Engineering, and Computer Science.

Additional Notes:
- Several universities offer graduate certificates or master's programs online in remote sensing; PhD programs are mostly campus-based.
- Prospective students should evaluate curriculum transparency, research environment, and facility access.

Sources:
- Edurank: https://edurank.org/engineering/remote-sensing/na/
- GeologyPrograms.com: https://www.geologyprograms.com/rankings/best-remote-sensing-programs/
- EliteDegrees.net: https://elitedegrees.net/remote-sensing/united-states/